# Horse Handicapping Model
### Version: 1.4
### Date of Last Edits: Sept 18, 2026
### Created by: John Little

In [71]:
## BLOCK 1: Inputs

import io
import tkinter as pd_tk
import numpy as np
import pandas as pd


def parse_odds_to_decimal(odds_val):
  """Converts fractional odds strings (e.g., '7/2', '5/2', '20') or floats into

  decimal values for comparison. Returns np.nan on failure.
  """
  if pd.isna(odds_val):
    return np.nan
  s = str(odds_val).strip()
  if not s or s.upper() == "NAN":
    return np.nan
  try:
    if "/" in s:
      num, den = s.split("/")
      return float(num) / float(den)
    return float(s)
  except Exception:
    return np.nan


def ingest_csv_clipboard(title_msg):
  """Ingests CSV-formatted data directly from the system clipboard,"""
  print("=" * 60)
  print(title_msg)
  print("=" * 60)

  try:
    root = pd_tk.Tk()
    root.withdraw()
    clip_text = root.clipboard_get()
    root.destroy()
    df = pd.read_csv(io.StringIO(clip_text))
  except Exception as e:
    print(
        f"Clipboard read error (is tkinter available?): {e}. Falling back to"
        " pd.read_clipboard..."
    )
    try:
      df = pd.read_clipboard(sep=",")
    except Exception as e2:
      print(f"Fallback failed: {e2}")
      return None

  if df is None or df.empty:
    print("Error: Clipboard is empty or could not be parsed.")
    return None

  df.columns = df.columns.str.strip().str.upper()
  return df


if __name__ == "__main__":
  # --------------------------------------------------------------------
  # 1. MAIN RUNNER DATA INGESTION & CLEANING
  # --------------------------------------------------------------------
  input(
      "PAUSE 1: Copy your RUNNER CSV from Gemini to clipboard, then press"
      " Enter..."
  )
  df_race = ingest_csv_clipboard("INGESTING MAIN RUNNER CSV FROM CLIPBOARD")

  if df_race is not None:
    primary_key = "POST" if "POST" in df_race.columns else "PROGRAM"
    if primary_key not in df_race.columns:
      primary_key = df_race.columns[0]

    df_race[primary_key] = pd.to_numeric(df_race[primary_key], errors="coerce")
    df_race = df_race.dropna(subset=[primary_key])
    df_race[primary_key] = df_race[primary_key].astype("Int64")
    df_race.set_index(primary_key, inplace=True, drop=False)

    # Clean & Format New Odds Fields
    if "ML_ODDS" in df_race.columns:
      df_race["ML_ODDS_DEC"] = df_race["ML_ODDS"].apply(parse_odds_to_decimal)
    else:
      df_race["ML_ODDS_DEC"] = np.nan

    if "LIVE_ODDS" in df_race.columns:
      df_race["LIVE_ODDS_DEC"] = df_race["LIVE_ODDS"].apply(
          parse_odds_to_decimal
      )
      df_race["LIVE_ODDS_DEC"] = df_race["LIVE_ODDS_DEC"].fillna(
          df_race["ML_ODDS_DEC"]
      )
    else:
      df_race["LIVE_ODDS_DEC"] = df_race["ML_ODDS_DEC"]

    # Clean & Format Run Style Fields
    if "RUN_STYLE" in df_race.columns:
      df_race["RUN_STYLE"] = (
          df_race["RUN_STYLE"].astype(str).str.strip().str.upper()
      )
    else:
      df_race["RUN_STYLE"] = "NA"

    if "RUN_STYLE_PTS" in df_race.columns:
      df_race["RUN_STYLE_PTS"] = (
          pd.to_numeric(df_race["RUN_STYLE_PTS"], errors="coerce")
          .fillna(0)
          .astype(int)
      )
    else:
      df_race["RUN_STYLE_PTS"] = 0

    print(
        f"\nSuccessfully ingested {len(df_race)} horses across"
        f" {len(df_race.columns)} columns!"
    )
    print("\nDataFrame Shape:", df_race.shape)
    print(df_race.head())

  # --------------------------------------------------------------------
  # 2. RACE STATS DATA INGESTION & BIAS BLENDING
  # --------------------------------------------------------------------
  print("\n" + "=" * 60)
  input(
      "PAUSE 2: Copy your RACE STATS CSV from Gemini to clipboard, then press"
      " Enter..."
  )
  df_stats = ingest_csv_clipboard("INGESTING RACE STATS CSV FROM CLIPBOARD")

  clean_breed = "Thoroughbred"
  clean_track = "Unknown Track"
  race_num = "1"
  active_speed_bias = 0.0
  post_iv_mapping = {}
  run_style_iv_mapping = {"E": 1.0, "E/P": 1.0, "P": 1.0, "S": 1.0}

  if df_stats is not None:
    if "BREED" in df_stats.columns:
      clean_breed = str(df_stats["BREED"].iloc[0]).strip().title()
    if "TRACK" in df_stats.columns:
      clean_track = str(df_stats["TRACK"].iloc[0]).strip().title()
    if "RACE_NUMBER" in df_stats.columns:
      race_num = str(df_stats["RACE_NUMBER"].iloc[0]).strip()
    elif "RACE_NUM" in df_stats.columns:
      race_num = str(df_stats["RACE_NUM"].iloc[0]).strip()

    df_stats["STAT_SET"] = df_stats["STAT_SET"].str.strip().str.title()

    week_rows = df_stats[df_stats["STAT_SET"] == "Week"]
    meet_rows = df_stats[df_stats["STAT_SET"] == "Meet"]

    if not week_rows.empty and not meet_rows.empty:
      week_data = week_rows.iloc[0]
      meet_data = meet_rows.iloc[0]
      week_races = week_data["RACES"]

      # 2-Tier Dynamic Blending Split Based on Sample Size (<15 vs >=15)
      if week_races < 15:
        w_week, w_meet = 0.00, 1.00
        blend_msg = (
            "Using 100% Meet stats (Week sample too small:"
            f" {int(week_races)} < 15 races)"
        )
      else:
        w_week, w_meet = 0.65, 0.35
        blend_msg = (
            "Blending 65% Week / 35% Meet (Week sample sufficient:"
            f" {int(week_races)} >= 15 races)"
        )

      print(f"\n[Rule Applied] {blend_msg}")

      active_speed_bias = float(
          (w_week * week_data["SPEED_BIAS"])
          + (w_meet * meet_data["SPEED_BIAS"])
      )

      # Post Position Impact Values (Rounded to nearest hundredth)
      post_iv_mapping = {
          "RAIL": round(
              float(
                  (w_week * week_data["IV_RAIL"])
                  + (w_meet * meet_data["IV_RAIL"])
              ),
              2,
          ),
          "1-3": round(
              float(
                  (w_week * week_data["IV_1TO3"])
                  + (w_meet * meet_data["IV_1TO3"])
              ),
              2,
          ),
          "4-7": round(
              float(
                  (w_week * week_data["IV_4TO7"])
                  + (w_meet * meet_data["IV_4TO7"])
              ),
              2,
          ),
          "8+": round(
              float(
                  (w_week * week_data["IV_8PLUS"])
                  + (w_meet * meet_data["IV_8PLUS"])
              ),
              2,
          ),
      }

      # Run Style Impact Values (Rounded to nearest hundredth)
      run_style_iv_mapping = {
          "E": round(
              float(
                  (w_week * week_data.get("IV_E", 1.0))
                  + (w_meet * meet_data.get("IV_E", 1.0))
              ),
              2,
          ),
          "E/P": round(
              float(
                  (w_week * week_data.get("IV_EP", 1.0))
                  + (w_meet * meet_data.get("IV_EP", 1.0))
              ),
              2,
          ),
          "P": round(
              float(
                  (w_week * week_data.get("IV_P", 1.0))
                  + (w_meet * meet_data.get("IV_P", 1.0))
              ),
              2,
          ),
          "S": round(
              float(
                  (w_week * week_data.get("IV_S", 1.0))
                  + (w_meet * meet_data.get("IV_S", 1.0))
              ),
              2,
          ),
      }

    elif not meet_rows.empty:
      meet_data = meet_rows.iloc[0]
      print(
          "\n[Rule Applied] Week data missing. Defaulting to 100% Meet stats."
      )
      active_speed_bias = float(meet_data["SPEED_BIAS"])

      post_iv_mapping = {
          "RAIL": round(float(meet_data["IV_RAIL"]), 2),
          "1-3": round(float(meet_data["IV_1TO3"]), 2),
          "4-7": round(float(meet_data["IV_4TO7"]), 2),
          "8+": round(float(meet_data["IV_8PLUS"]), 2),
      }

      run_style_iv_mapping = {
          "E": round(float(meet_data.get("IV_E", 1.0)), 2),
          "E/P": round(float(meet_data.get("IV_EP", 1.0)), 2),
          "P": round(float(meet_data.get("IV_P", 1.0)), 2),
          "S": round(float(meet_data.get("IV_S", 1.0)), 2),
      }
    else:
      print("Error: Could not locate Meet stats in the scraped table.")

    print(f"Active Blended Speed Bias: {active_speed_bias:.3f}")
    print(f"Blended Post Impact Value Map: {post_iv_mapping}")
    print(f"Blended Run Style Impact Value Map: {run_style_iv_mapping}")

PAUSE 1: Copy your RUNNER CSV from Gemini to clipboard, then press Enter... PROGRAM,HORSE_NAME,POST,ML_ODDS,LIVE_ODDS,RUN_STYLE,RUN_STYLE_PTS,PRM_PWR,AVG_SPD,BACK_SPD,SPD_LR,AVG_CLS,LAST_CLS,AVG_DIST_SPD,BEST_SPD,W_JKY,W_TRN,E1,E2,LP,DAYS_OFF 1,"Joy Boy",1,"10","8","P",4,106.2,64,0,66,107.9,107,0,0,13.6,21.3,0,0,0,16 2,"Agendum",2,"4","9/2","P",5,106.7,65,76,76,107.5,109,72,68,7.1,15.1,83,84,68,19 3,"Coach Michael",3,"10","9","E",5,101.3,63,0,61,107.1,107,0,0,13.6,19.5,0,0,0,22 4,"D'oro Cavallo",4,"3","9/2","S",0,109.2,65,75,75,106.5,108,71,75,17.4,8.6,71,72,78,11 5,"Mr. Handsome",5,"12","14","NA",0,79.0,27,0,56,101.8,104,0,0,13.1,11.9,0,0,0,29 6,"Seawise",6,"2","2","E/P",6,107.2,74,75,79,109.0,110,74,75,9.1,13.7,88,85,68,181 7,"Chance On Me",7,"30","24","E/P",5,73.6,51,63,57,102.3,102,63,63,2.0,5.4,81,73,64,20 8,"Dufoof",8,"9/2","7/2","NA",0,104.2,62,66,66,108.8,109,62,57,14.3,12.5,60,51,80,313


INGESTING MAIN RUNNER CSV FROM CLIPBOARD

Successfully ingested 8 horses across 23 columns!

DataFrame Shape: (8, 23)
      PROGRAM     HORSE_NAME  POST ML_ODDS LIVE_ODDS RUN_STYLE  RUN_STYLE_PTS  \
POST                                                                            
1           1        Joy Boy     1      10         8         P              4   
2           2        Agendum     2       4       9/2         P              5   
3           3  Coach Michael     3      10         9         E              5   
4           4  D'oro Cavallo     4       3       9/2         S              0   
5           5   Mr. Handsome     5      12        14       NaN              0   

      PRM_PWR  AVG_SPD  BACK_SPD  ...  AVG_DIST_SPD  BEST_SPD  W_JKY  W_TRN  \
POST                              ...                                         
1       106.2       64         0  ...             0         0   13.6   21.3   
2       106.7       65        76  ...            72        68    7.1   15.1  

PAUSE 2: Copy your RACE STATS CSV from Gemini to clipboard, then press Enter... BREED,TRACK,RACE_NUMBER,STAT_SET,RACES,SPEED_BIAS,IV_RAIL,IV_1to3,IV_4to7,IV_8plus,IV_E,IV_EP,IV_P,IV_S Thoroughbred,Delaware Park,6,Meet,5,0.400,1.77,1.18,0.95,0.83,0.95,1.13,1.26,0.55 Thoroughbred,Delaware Park,6,Week,1,0.000,0.00,0.00,0.00,3.33,0.00,0.00,0.00,5.06


INGESTING RACE STATS CSV FROM CLIPBOARD

[Rule Applied] Using 100% Meet stats (Week sample too small: 1 < 15 races)
Active Blended Speed Bias: 0.400
Blended Post Impact Value Map: {'RAIL': 1.77, '1-3': 1.18, '4-7': 0.95, '8+': 0.83}
Blended Run Style Impact Value Map: {'E': 0.95, 'E/P': 1.13, 'P': 1.26, 'S': 0.55}


In [73]:
#COMBINED BLOCKS 2-5 TO RUN IN A SINGLE BLOCK
# BLOCK 2: Weight Setup & Scaling (based on race stats)

# Master weight definitions (Scaled to 100% Base Weights)
weights_dict = {
    'Thoroughbred': {
        'W_Speed': 0.16,
        'W_Power': 0.09,
        'W_Class': 0.14,
        'W_Distance': 0.11,
        'W_Driver': 0.06,
        'W_Trainer': 0.06,
        'W_Early': 0.15,
        'W_Finish': 0.10,
        'W_Recency': 0.03,
        'W_Course': 0.02,
        'W_Market': 0.08,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
    'Harness': {
        'W_Speed': 0.13,
        'W_Power': 0.07,
        'W_Class': 0.11,
        'W_Distance': 0.04,
        'W_Driver': 0.18,
        'W_Trainer': 0.06,
        'W_Early': 0.18,
        'W_Finish': 0.08,
        'W_Recency': 0.04,
        'W_Course': 0.03,
        'W_Market': 0.08,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
    'Quarter Horse': {
        'W_Speed': 0.22,
        'W_Power': 0.10,
        'W_Class': 0.09,
        'W_Distance': 0.02,
        'W_Driver': 0.07,
        'W_Trainer': 0.07,
        'W_Early': 0.30,
        'W_Finish': 0.00,
        'W_Recency': 0.02,
        'W_Course': 0.00,
        'W_Market': 0.11,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
    'Churchill Downs': {
        'W_Speed': 0.16,
        'W_Power': 0.09,
        'W_Class': 0.17,
        'W_Distance': 0.11,
        'W_Driver': 0.08,
        'W_Trainer': 0.08,
        'W_Early': 0.11,
        'W_Finish': 0.12,
        'W_Recency': 0.03,
        'W_Course': 0.02,
        'W_Market': 0.03,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
}

weights_df = pd.DataFrame(weights_dict)

# 1. Select Base Weights
if clean_track == 'Churchill Downs':
  active_weights = weights_df['Churchill Downs'].copy()
  weight_mode_msg = 'SPECIAL CHURCHILL DOWNS WEIGHTS'
else:
  if clean_breed in weights_df.columns:
    active_weights = weights_df[clean_breed].copy()
    weight_mode_msg = f'Standard {clean_breed} Base Weights'
  else:
    raise ValueError(f"Breed '{clean_breed}' not found in model weights.")

# 2. Detect Meet Sample Size
meet_var_candidates = [
    'meet_races',
    'meet_race_count',
    'races_meet',
    'MEET_RACES',
    'races_in_meet',
    'meet_count',
]
active_meet_count = 10  # Explicit default for low-sample meets unless overridden

for var in meet_var_candidates:
  if var in locals():
    active_meet_count = locals()[var]
    break

# Set blending weights based on 3-tier sample size rule
if active_meet_count < 6:
  meet_weight = 0.00
  base_weight = 1.00
  sample_scale_factor = 0.00
elif active_meet_count <= 15:
  meet_weight = 0.10
  base_weight = 0.90
  sample_scale_factor = 0.10
else:
  meet_weight = 0.65
  base_weight = 0.35
  sample_scale_factor = 1.00

# 3. Dynamic Early Pace Scaling
bias_decimal = (
    active_speed_bias / 100.0 if active_speed_bias > 1.0 else active_speed_bias
)
base_w_early = active_weights['W_Early']

if bias_decimal > 0.55:
  raw_early_boost = (bias_decimal - 0.55) * 0.5
  early_pace_boost = raw_early_boost * sample_scale_factor
  active_weights['W_Early'] = base_w_early + early_pace_boost

  early_msg = (
      f"Boosted from {base_w_early:.1%} to {active_weights['W_Early']:.1%} "
      f'(Dampened to {sample_scale_factor:.0%} weight | Meet Races:'
      f' {active_meet_count})'
  )
else:
  early_msg = (
      f'Kept at Base {base_w_early:.1%} (Speed Bias: {bias_decimal:.1%} <= 55%)'
  )

# Unpack scalar variables
w_speed = active_weights['W_Speed']
w_power = active_weights['W_Power']
w_class = active_weights['W_Class']
w_distance = active_weights['W_Distance']
w_driver = active_weights['W_Driver']
w_trainer = active_weights['W_Trainer']
w_early = active_weights['W_Early']
w_finish = active_weights['W_Finish']
w_recency = active_weights['W_Recency']
w_course = active_weights['W_Course']
w_market = active_weights['W_Market']
w_postbias = active_weights['W_PostBias']
w_style = active_weights['W_Style']

#-------------------------------------------------------------------------------------------------------------------------------
# BLOCK 3: Dynamic Post Bias Engine & Confirmation

# -------------------------------------------------------------------------------------------------------------------------------
# BLOCK 3: Dynamic Post Bias Engine & Confirmation

print("\n" + "=" * 60)
print("CALCULATING DYNAMIC POST & STYLE BIAS BONUSES")
print("=" * 60)

# 1. Multiplier Setup by Breed
if clean_breed == "Harness":
  post_iv_multiplier = 7.0
  style_iv_multiplier = 7.0
elif clean_breed == "Quarter Horse":
  post_iv_multiplier = 4.0
  style_iv_multiplier = 5.0
else:  # Thoroughbred
  post_iv_multiplier = 6.0
  style_iv_multiplier = 6.0

MAX_BIAS_CAP = 5.00


def calculate_post_bonus(post_cat, iv_map, multiplier, max_cap=MAX_BIAS_CAP):
  iv = iv_map.get(post_cat, 1.00)
  if iv <= 0:  # Safety guard for missing track data
    iv = 1.00

  iv_diff = max(0.0, iv - 1.00)  # Positive bonus ONLY (No negative penalties)

  if iv_diff >= 0.03:
    raw_bonus = iv_diff * multiplier
    return round(min(max_cap, raw_bonus), 2)
  return 0.0


def calculate_style_bonus(
    style_code, style_pts, iv_map, multiplier, max_cap=MAX_BIAS_CAP
):
  code = str(style_code).strip().upper()
  iv = iv_map.get(code, 1.00)
  if iv <= 0:  # Safety guard for missing track data
    iv = 1.00

  iv_diff = max(0.0, iv - 1.00)  # Positive bonus ONLY (No negative penalties)

  if iv_diff >= 0.03:
    try:
      pts_factor = min(max(float(style_pts) / 8.0, 0.0), 1.0)
    except (ValueError, TypeError):
      pts_factor = 0.50

    raw_bonus = iv_diff * multiplier * pts_factor
    return round(min(max_cap, raw_bonus), 2)
  return 0.0


# 2. Apply Bonuses to DataFrame
if "df_race" in locals() and df_race is not None:

  # SAMPLE SIZE GUARD: Zero out bonuses if meet sample is under 15 races
  if active_meet_count < 15:
    print(
        f"\n[NOTICE] Low Meet Sample Size ({active_meet_count} races < 15"
        " threshold). All Post & Style bonuses set to 0.00."
    )
    df_race["POST_CAT"] = "N/A"
    df_race["POST_IV"] = 1.00
    df_race["POST_BONUS"] = 0.00
    df_race["STYLE_IV"] = 1.00
    df_race["STYLE_BONUS"] = 0.00
  else:
    primary_post_col = "POST" if "POST" in df_race.columns else "PROGRAM"

    # Apply Post Position Bonuses
    df_race["POST_CAT"] = df_race[primary_post_col].apply(get_post_category)
    df_race["POST_IV"] = df_race["POST_CAT"].apply(
        lambda cat: (
            1.00
            if post_iv_mapping.get(cat, 1.00) <= 0
            else post_iv_mapping.get(cat, 1.00)
        )
    )
    df_race["POST_BONUS"] = df_race["POST_CAT"].apply(
        lambda cat: calculate_post_bonus(
            cat, post_iv_mapping, post_iv_multiplier
        )
    )

    # Apply Running Style Bonuses
    if "RUN_STYLE" in df_race.columns:
      df_race["STYLE_IV"] = df_race["RUN_STYLE"].apply(
          lambda style: (
              1.00
              if run_style_iv_mapping.get(style, 1.00) <= 0
              else run_style_iv_mapping.get(style, 1.00)
          )
      )
      df_race["STYLE_BONUS"] = df_race.apply(
          lambda row: calculate_style_bonus(
              row.get("RUN_STYLE", "NA"),
              row.get("RUN_STYLE_PTS", 0),
              run_style_iv_mapping,
              style_iv_multiplier,
          ),
          axis=1,
      )
    else:
      df_race["STYLE_IV"] = 1.00
      df_race["STYLE_BONUS"] = 0.00

  print("\nPost & Run Style Bias Bonus Summary by Horse:")
  print(
      df_race[[
          "POST",
          "POST_CAT",
          "POST_BONUS",
          "RUN_STYLE",
          "RUN_STYLE_PTS",
          "STYLE_IV",
          "STYLE_BONUS",
      ]]
  )

#-------------------------------------------------------------------------------------------------------------------------------
## BLOCK 4: Composite Sub-Score & Weighted Final Score Engine

import numpy as np
import pandas as pd

df_calc = df_race.copy()

# --------------------------------------------------------------------
# 0. MISSING DATA IMPUTATION ENGINE (Field-Average Imputation)
# --------------------------------------------------------------------

# A. Categorical & Odds Imputation
if 'LIVE_ODDS_DEC' in df_calc.columns:
  df_calc['LIVE_ODDS_DEC'] = (
      pd.to_numeric(df_calc['LIVE_ODDS_DEC'], errors='coerce')
      .replace(0, np.nan)
      .fillna(df_calc.get('ML_ODDS_DEC', np.nan))
  )

if 'RUN_STYLE' in df_calc.columns:
  df_calc['RUN_STYLE'] = df_calc['RUN_STYLE'].fillna('NA')
if 'RUN_STYLE_PTS' in df_calc.columns:
  df_calc['RUN_STYLE_PTS'] = pd.to_numeric(
      df_calc['RUN_STYLE_PTS'], errors='coerce'
  ).fillna(0)

# B. Numerical Imputation (Non-Zero Race Average)
impute_num_cols = [
    'PRM_PWR',
    'PRIME_POWER',
    'AVG_SPD',
    'BACK_SPD',
    'SPD_LR',
    'AVG_CLS',
    'CLASS_RATING',
    'LAST_CLS',
    'AVG_DIST_SPD',
    'BEST_SPD',
    'W_JKY',
    'JOCKEY_WIN_PCT',
    'W_TRN',
    'TRAINER_WIN_PCT',
    'E1',
    'E2',
    'EARLY_PACE',
    'LP',
    'LATE_PACE',
    'DAYS_OFF',
    'TRACK_WIN_PCT',
    'COURSE_HISTORY',
    'TRACK_ITM',
]

for col in impute_num_cols:
  if col in df_calc.columns:
    # Convert strings/dash values to numeric, turning non-numerics into NaN
    df_calc[col] = pd.to_numeric(df_calc[col], errors='coerce')

    # Identify valid active entries (> 0)
    valid_entries = df_calc[col].notna() & (df_calc[col] > 0)

    if valid_entries.any():
      # Calculate non-zero mean across active runners in this race
      field_avg = round(df_calc.loc[valid_entries, col].mean(), 1)
      # Replace NaN, 0, or negative values with the field average
      df_calc[col] = df_calc[col].apply(
          lambda x: field_avg if pd.isna(x) or x <= 0 else x
      )
    else:
      df_calc[col] = df_calc[col].fillna(0.0)

# Final safety net for any unhandled numeric columns
numeric_cols = df_calc.select_dtypes(include=['float64', 'int64']).columns
df_calc[numeric_cols] = df_calc[numeric_cols].fillna(0)

# --------------------------------------------------------------------
# 1. CALCULATE RAW COMPOSITE SUB-SCORES
# --------------------------------------------------------------------

# 1. Speed Rating
if 'AVG_DIST_SPD' in df_calc.columns and 'SPD_LR' in df_calc.columns:
  df_calc['COMP_SPEED'] = (0.70 * df_calc['AVG_DIST_SPD']) + (
      0.30 * df_calc['SPD_LR']
  )
else:
  df_calc['COMP_SPEED'] = df_calc.get('SPD_LR', df_calc.get('AVG_DIST_SPD', 0))

# 2. Prime Power Rating
df_calc['COMP_POWER'] = df_calc.get('PRM_PWR', df_calc.get('PRIME_POWER', 0))

# 3. Class & Form
df_calc['COMP_CLASS'] = df_calc.get(
    'AVG_CLS', df_calc.get('CLASS_RATING', df_calc.get('LAST_CLS', 0))
)

# 4. Distance & Surface
df_calc['COMP_DISTANCE'] = df_calc.get(
    'AVG_DIST_SPD', df_calc.get('BEST_SPD', 0)
)

# 5. Jockey / Driver
df_calc['COMP_DRIVER'] = df_calc.get('W_JKY', df_calc.get('JOCKEY_WIN_PCT', 0))

# 6. Trainer
df_calc['COMP_TRAINER'] = df_calc.get(
    'W_TRN', df_calc.get('TRAINER_WIN_PCT', 0)
)

# 7. Early Pace
if 'E1' in df_calc.columns and 'E2' in df_calc.columns:
  df_calc['COMP_EARLY'] = (0.50 * df_calc['E1']) + (0.50 * df_calc['E2'])
else:
  df_calc['COMP_EARLY'] = df_calc.get(
      'E2', df_calc.get('E1', df_calc.get('EARLY_PACE', 0))
  )

# 8. Finish Pace
df_calc['COMP_FINISH'] = df_calc.get('LP', df_calc.get('LATE_PACE', 0))

# 9. Days Off / Recency
if 'DAYS_OFF' in df_calc.columns:

  def score_recency(days):
    try:
      d = float(days)
      if 14 <= d <= 45:
        return 100.0
      elif d < 14:
        return 85.0
      elif 46 <= d <= 90:
        return 70.0
      else:
        return 50.0
    except (ValueError, TypeError):
      return 75.0

  df_calc['COMP_RECENCY'] = df_calc['DAYS_OFF'].apply(score_recency)
else:
  df_calc['COMP_RECENCY'] = df_calc.get('RECENCY', 75.0)

# 10. Course History
df_calc['COMP_COURSE'] = df_calc.get(
    'TRACK_WIN_PCT', df_calc.get('COURSE_HISTORY', df_calc.get('TRACK_ITM', 0))
)

# 11. Market Odds / Sentiment (Implied Probability normalized to 0-100 scale)
if 'LIVE_ODDS_DEC' in df_calc.columns:
  df_calc['COMP_MARKET'] = df_calc['LIVE_ODDS_DEC'].apply(
      lambda odds: (1.0 / (odds + 1.0)) * 100.0 if odds > 0 else 0.0
  )
elif 'ML_ODDS_DEC' in df_calc.columns:
  df_calc['COMP_MARKET'] = df_calc['ML_ODDS_DEC'].apply(
      lambda odds: (1.0 / (odds + 1.0)) * 100.0 if odds > 0 else 0.0
  )
else:
  df_calc['COMP_MARKET'] = 0.0

# --------------------------------------------------------------------
# 2. APPLY WEIGHTS & CALCULATE FINAL SCORES
# --------------------------------------------------------------------

df_calc['SCORE_SPEED'] = df_calc['COMP_SPEED'] * w_speed
df_calc['SCORE_POWER'] = df_calc['COMP_POWER'] * w_power
df_calc['SCORE_CLASS'] = df_calc['COMP_CLASS'] * w_class
df_calc['SCORE_DISTANCE'] = df_calc['COMP_DISTANCE'] * w_distance
df_calc['SCORE_DRIVER'] = df_calc['COMP_DRIVER'] * w_driver
df_calc['SCORE_TRAINER'] = df_calc['COMP_TRAINER'] * w_trainer
df_calc['SCORE_EARLY'] = df_calc['COMP_EARLY'] * w_early
df_calc['SCORE_FINISH'] = df_calc['COMP_FINISH'] * w_finish
df_calc['SCORE_RECENCY'] = df_calc['COMP_RECENCY'] * w_recency
df_calc['SCORE_COURSE'] = df_calc['COMP_COURSE'] * w_course
df_calc['SCORE_MARKET'] = df_calc['COMP_MARKET'] * w_market

weighted_score_cols = [
    'SCORE_SPEED',
    'SCORE_POWER',
    'SCORE_CLASS',
    'SCORE_DISTANCE',
    'SCORE_DRIVER',
    'SCORE_TRAINER',
    'SCORE_EARLY',
    'SCORE_FINISH',
    'SCORE_RECENCY',
    'SCORE_COURSE',
    'SCORE_MARKET',
]
df_calc['BASE_SKILL_SCORE'] = df_calc[weighted_score_cols].sum(axis=1)

# Add post position & run style bias bonuses
df_calc['FINAL_SCORE'] = (round(
    df_calc['BASE_SKILL_SCORE']
    + df_calc['POST_BONUS']
    + df_calc.get('STYLE_BONUS', 0.0), 2)
)

df_calc['RANK'] = (
    df_calc['FINAL_SCORE'].rank(ascending=False, method='min').astype(int)
)
df_calc.sort_values(by='RANK', inplace=True)

# --------------------------------------------------------------------
# DISPLAY FINAL MODEL RANKINGS
# --------------------------------------------------------------------
print('=' * 75)
print(f'FINAL MODEL RANKINGS | {clean_track} - RACE #{race_num}')
print('=' * 75)

display_fields = [
    c
    for c in [
        'RANK',
        'PROGRAM',
        'HORSE_NAME',
        'RUN_STYLE',
        'BASE_SKILL_SCORE',
        'POST_BONUS',
        'STYLE_BONUS',
        'FINAL_SCORE',
        'POST',
    ]
    if c in df_calc.columns
]
print(df_calc[display_fields].to_string(index=False))
#-------------------------------------------------------------------------------------------------------------------------------
# BLOCK 5: Wager Recommendations

if 'df_calc' in locals() and len(df_calc) >= 3:
    top_field = df_calc.head(5).copy().reset_index(drop=True)
    num_horses = len(top_field)
    total_starters = len(df_calc)
    p_col = "PROGRAM" if "PROGRAM" in top_field.columns else "POST"

    top_nums = top_field[p_col].astype(str).tolist()
    top_scores = top_field["FINAL_SCORE"].tolist()

    # Pad arrays if fewer than 5 horses in race
    while len(top_nums) < 5:
        top_nums.append("N/A")
        top_scores.append(0.0)

    s1, s2, s3, s4, s5 = top_scores[:5]

    # Calculate Score Gaps across top 5
    gap_12 = round(s1 - s2, 2)
    gap_23 = round(s2 - s3, 2)
    gap_34 = round(s3 - s4, 2)
    gap_13 = round(s1 - s3, 2)
    gap_14 = round(s1 - s4, 2)

    # --------------------------------------------------------------------
    # 1. CHURCHILL DOWNS ODD/EVEN SPECIAL BET CHECK
    # --------------------------------------------------------------------
    churchill_odd_even_rec = None

    if clean_track == "Churchill Downs" and total_starters >= 6:
        all_program_nums = []
        for p in df_calc[p_col]:
            try:
                clean_p = int(''.join(filter(str.isdigit, str(p))))
                all_program_nums.append(clean_p)
            except ValueError:
                continue

        odd_count = sum(1 for n in all_program_nums if n % 2 != 0)
        even_count = sum(1 for n in all_program_nums if n % 2 == 0)

        if odd_count >= 3 and even_count >= 3 and (gap_13 < 3.0 or gap_14 < 3.0):
            odd_score_sum = 0.0
            even_score_sum = 0.0

            for i in range(min(5, num_horses)):
                try:
                    num_val = int(''.join(filter(str.isdigit, str(top_nums[i]))))
                    if num_val % 2 != 0:
                        odd_score_sum += top_scores[i]
                    else:
                        even_score_sum += top_scores[i]
                except ValueError:
                    continue

            preferred_side = "ODD" if odd_score_sum >= even_score_sum else "EVEN"
            side_runners = [
                f"#{top_nums[i]}" for i in range(min(5, num_horses))
                if top_nums[i] != "N/A" and (
                    (int(''.join(filter(str.isdigit, str(top_nums[i])))) % 2 != 0) if preferred_side == "ODD"
                    else (int(''.join(filter(str.isdigit, str(top_nums[i])))) % 2 == 0)
                )
            ]

            churchill_odd_even_rec = (
                f"CHURCHILL ODD/EVEN WAGER: Bet [{preferred_side}]\n"
                f"      Reason: Contested top group favors {preferred_side} runners ({', '.join(side_runners)}) "
                f"with a score weight of {max(odd_score_sum, even_score_sum):.2f} vs {min(odd_score_sum, even_score_sum):.2f}."
            )

    # --------------------------------------------------------------------
    # 2. SCENARIO IDENTIFICATION & WAGER FORMULATION
    # --------------------------------------------------------------------
    
    # Chaos / Ultra-Tight Field -> 3-Horse Exacta Box for affordable coverage
    if gap_14 < 3.0 and num_horses >= 4:
        scenario = "Ultra-Tight Field / High Chaos (Top 4 within 3.0 pts)"
        option_a = f"PASS / NO BET (Or Value WIN Wager on highest live odds among #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]})"
        option_b = f"3-Horse Exacta Box: #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]} (6 combos / low cost)"

    # Clear Dominant Winner (Gap 1-2 >= 5.0)
    elif gap_12 >= 5.0:
        scenario = f"Dominant Standout (#{top_nums[0]} holds >5.0 pt lead)"
        option_a = f"WIN Wager on #{top_nums[0]}"
        if gap_23 < 3.0 and gap_34 >= 3.0:
            option_b = f"Straight Exacta: #{top_nums[0]} / #{top_nums[1]}, #{top_nums[2]}"
        else:
            option_b = f"Straight Exacta: #{top_nums[0]} / #{top_nums[1]} (Or Exacta Key: #{top_nums[0]} / #{top_nums[1]}, #{top_nums[2]}, #{top_nums[3]})"

    # Competitive Top Duo (Gap 1-2 < 5.0 AND Gap 2-3 >= 3.0)
    elif gap_12 < 5.0 and gap_23 >= 3.0:
        scenario = f"Competitive Top Duo (#{top_nums[0]} & #{top_nums[1]} separated from field)"
        option_a = f"WIN Wager on #{top_nums[0]} (Or PLACE Wager on #{top_nums[1]})"
        option_b = f"Exacta Box: #{top_nums[0]}, #{top_nums[1]}"

    # Volatile Top Trio (Gap 1-3 < 3.0 and Gap 1-4 >= 3.0)
    elif gap_13 < 3.0:
        scenario = f"Volatile Top Group (Top 3 within 3.0 pts: #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]})"
        option_a = f"PLACE / SHOW Wager on highest live odds among #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]}"
        option_b = f"Trifecta Box: #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]} (6 combos / $3.00 total at $0.50 base)"

    # Standard Competitive Field
    else:
        scenario = "Standard Competitive Field"
        option_a = f"WIN / PLACE Wager on #{top_nums[0]}"
        option_b = f"Straight Exacta Wheel: #{top_nums[0]} / #{top_nums[1]}, #{top_nums[2]}"

    # --------------------------------------------------------------------
    # DASHBOARD DISPLAY
    # --------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("MODEL WAGER RECOMMENDATION DASHBOARD")
    print("=" * 70)
    print(f"Race Scenario Identified: {scenario}")
    print("-" * 70)
    print("Top 5 Field Distribution & Gap Analysis:")
    for i in range(min(5, num_horses)):
        print(f"  Rank {i+1}: #{top_nums[i]} - Score: {top_scores[i]:.2f}" + (f" (Gap to #1: -{s1 - top_scores[i]:.2f})" if i > 0 else " (Leader)"))
    print("-" * 70)
    print(f"OPTION A (Straight Wager - Single Horse Focus):")
    print(f"  -> {option_a}\n")
    print(f"OPTION B (Exotic Wager - High Payout Alternative):")
    print(f"  -> {option_b}")
    
    if churchill_odd_even_rec:
        print("\nOPTION C (Special Track Option - Churchill Downs):")
        print(f"  -> {churchill_odd_even_rec}")
        
    print("=" * 70)
else:
    print("Error: `df_calc` not found or insufficient horses to generate wager recommendations.")


CALCULATING DYNAMIC POST & STYLE BIAS BONUSES

[NOTICE] Low Meet Sample Size (10 races < 15 threshold). All Post & Style bonuses set to 0.00.

Post & Run Style Bias Bonus Summary by Horse:
      POST POST_CAT  POST_BONUS RUN_STYLE  RUN_STYLE_PTS  STYLE_IV  \
POST                                                                 
1        1      N/A         0.0         P              4       1.0   
2        2      N/A         0.0         P              5       1.0   
3        3      N/A         0.0         E              5       1.0   
4        4      N/A         0.0         S              0       1.0   
5        5      N/A         0.0       NaN              0       1.0   
6        6      N/A         0.0       E/P              6       1.0   
7        7      N/A         0.0       E/P              5       1.0   
8        8      N/A         0.0       NaN              0       1.0   

      STYLE_BONUS  
POST               
1             0.0  
2             0.0  
3             0.0  
4        